In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import matplotlib.pyplot as plt

# 1. 加载数据
print("正在加载数据...")
# 修正文件路径，使用相同的平滑数据
df_original = pd.read_excel('../../data/pinghua_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# 6. 定义搜索空间 - 针对LightGBM的参数
dimensions = [
    Integer(100, 500, name='n_estimators'),
    Integer(3, 15, name='max_depth'),
    Integer(10, 100, name='num_leaves'),
    Real(0.01, 0.2, name='learning_rate'),
    Real(0.6, 1.0, name='subsample'),
    Real(0.6, 1.0, name='colsample_bytree'),
    Integer(5, 30, name='min_child_samples')
]

# 7. 定义目标函数 - 使用测试集R²作为优化目标
@use_named_args(dimensions)
def objective(n_estimators, max_depth, num_leaves, learning_rate, subsample, colsample_bytree, min_child_samples):
    # 组合参数
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'num_leaves': num_leaves,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'min_child_samples': min_child_samples,
        'verbose': -1,
        'random_state': 42
    }
    
    # 使用参数创建模型
    base_model = lgb.LGBMRegressor(**params)
    model = MultiOutputRegressor(base_model)
    
    # 在训练集上训练模型
    model.fit(X_train_scaled, y_train)
    
    # 在测试集上预测
    y_pred = model.predict(X_test_scaled)
    
    # 计算测试集上每个目标变量的R²
    r2_scores = []
    for idx in range(len(target_vars)):
        r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
        r2_scores.append(r2)
    
    # 计算平均R²并取负值（因为优化器是最小化函数）
    mean_r2 = np.mean(r2_scores)
    
    # 记录当前迭代的参数和R²
    print(f"参数: n_est={n_estimators}, depth={max_depth}, leaves={num_leaves}, lr={learning_rate:.3f}, subsample={subsample:.2f}, colsample={colsample_bytree:.2f}, min_child={min_child_samples} - 平均R²: {mean_r2:.4f}")
    
    return -mean_r2  # 返回负R²，因为我们要最大化R²，而优化器是最小化函数

# 8. 执行贝叶斯优化
print("\n开始基于测试集R²的LightGBM贝叶斯参数优化...")
result = gp_minimize(
    objective,
    dimensions=dimensions,
    n_calls=30,  # 迭代次数
    n_random_starts=10,  # 随机搜索起点数
    random_state=42,
    verbose=True,
    n_jobs=-1
)


# 10. 提取最佳参数
best_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'random_state': 42,
    'verbose': -1,
    'n_estimators': result.x[0],
    'max_depth': result.x[1],
    'num_leaves': result.x[2],
    'learning_rate': result.x[3],
    'subsample': result.x[4],
    'colsample_bytree': result.x[5],
    'min_child_samples': result.x[6]
}

print("\n最优参数:")
for param, value in best_params.items():
    if param not in ['objective', 'metric', 'random_state', 'verbose']:
        print(f"{param}: {value}")

# 11. 使用最优参数创建新模型
print("\n用最优参数训练最终模型...")
best_model = MultiOutputRegressor(lgb.LGBMRegressor(**best_params))

# 12. 用原始训练数据训练最终模型
best_model.fit(X_train_scaled, y_train)
print("模型训练完成！")

# 13. 在测试集上进行预测
y_pred = best_model.predict(X_test_scaled)

# 14. 评估最终模型性能
print("\n优化后模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 15. 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")

# 16. 输出贝叶斯优化的最佳结果
print("\n贝叶斯优化最佳结果:")
print(f"最佳R²: {-result.fun:.4f}")  # 去掉负号，因为我们优化时是负R²
print(f"优化迭代次数: {len(result.func_vals)}")

# 17. 为比较创建一个基准模型（使用Lightgbm.ipynb中的默认参数作为基准）
print("\n训练对照模型...")
baseline_params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 100,
    'max_depth': 29,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'subsample': 0.8,
    'min_child_samples': 3,
    'subsample_freq': 5,
    'verbose': -1,
    'n_jobs': -1,
    'n_estimators': 500,
    'random_state': 42
}

baseline_model = MultiOutputRegressor(lgb.LGBMRegressor(**baseline_params))

# 用原始训练数据训练基准模型
baseline_model.fit(X_train_scaled, y_train)

# 在测试集上评估基准模型
y_pred_baseline = baseline_model.predict(X_test_scaled)

# 计算基准模型指标
baseline_r2 = 0
baseline_rmse = 0
for idx in range(len(target_vars)):
    r2 = r2_score(y_test.iloc[:, idx], y_pred_baseline[:, idx])
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred_baseline[:, idx])
    rmse = np.sqrt(mse)
    baseline_r2 += r2
    baseline_rmse += rmse

baseline_r2 /= len(target_vars)
baseline_rmse /= len(target_vars)

print(f"\n基准模型平均R²: {baseline_r2:.4f}")
print(f"基准模型平均RMSE: {baseline_rmse:.4f}")
print(f"优化模型平均R²: {overall_r2:.4f}")
print(f"优化模型平均RMSE: {overall_rmse:.4f}")
print(f"R²改进: {(overall_r2 - baseline_r2):.4f}")
print(f"RMSE改进: {(baseline_rmse - overall_rmse):.4f}")
